# MLflow Models: Basic Model Logging and Loading

In this notebook we will cover:
1. What is MLflow Model
2. Basic logging of scikit-learn model  
3. Loading and using saved model
4. MLflow Model structure

## What is MLflow Model?

**MLflow Model** is a standard format for packaging ML models that:
- Is framework-independent
- Includes metadata and dependencies
- Simplifies deployment to different platforms
- Supports different "flavors" (sklearn, pytorch, tensorflow, etc.)

## 1. Import Libraries

In [ ]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import sys
import os

sys.path.append('../src')
from data_loader import load_sample_data
from model_inspector import inspect_model_structure, export_model

print(f"MLflow version: {mlflow.__version__}")

MLflow version: 3.4.0


## 2. Data Preparation

In [2]:
X_train, X_test, y_train, y_test, feature_names, target_names = load_sample_data('iris')

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Features: {feature_names}")
print(f"Target classes: {target_names}")

Training set size: (120, 4)
Test set size: (30, 4)
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target classes: ['setosa' 'versicolor' 'virginica']


## 3. Basic Model Logging

The simplest way to save a model is to use `mlflow.sklearn.log_model()`

In [3]:
mlflow.set_tracking_uri("file:./mlruns")
experiment_name = "01_Basic_Model_Logging"
mlflow.set_experiment(experiment_name)

print(f"Experiment: {experiment_name}")

2026/02/15 01:44:03 INFO mlflow.tracking.fluent: Experiment with name '01_Basic_Model_Logging' does not exist. Creating a new experiment.


Experiment: 01_Basic_Model_Logging


In [4]:
with mlflow.start_run(run_name="basic_rf_model") as run:
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("accuracy", accuracy)
    
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model"
    )
    
    run_id = run.info.run_id
    print(f"Run ID: {run_id}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"\nModel logged to: runs:/{run_id}/model")

2026/02/15 01:44:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/15 01:44:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run ID: 929953b4db2f4eb791b7408476d192dc
Accuracy: 0.9333

Model logged to: runs:/929953b4db2f4eb791b7408476d192dc/model


## 4. Loading and Using Model

### Method 1: Load as scikit-learn model

In [5]:
model_uri = f"runs:/{run_id}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)

predictions = loaded_model.predict(X_test)
print(f"Predictions shape: {predictions.shape}")
print(f"First 5 predictions: {predictions[:5]}")
print(f"Actual labels: {y_test[:5]}")

Predictions shape: (30,)
First 5 predictions: [0 2 1 1 0]
Actual labels: [0 2 1 1 0]


### Method 2: Load as Python Function (universal)

In [6]:
loaded_pyfunc = mlflow.pyfunc.load_model(model_uri)

X_test_df = pd.DataFrame(X_test, columns=feature_names)
pyfunc_predictions = loaded_pyfunc.predict(X_test_df)

print(f"PyFunc predictions shape: {pyfunc_predictions.shape}")
print(f"First 5 predictions: {pyfunc_predictions[:5]}")

PyFunc predictions shape: (30,)
First 5 predictions: [0 2 1 1 0]


c:\Learn\github\mlpops-basics\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


## 5. Exploring MLflow Model Structure

In [7]:
import mlflow.artifacts

local_path = mlflow.artifacts.download_artifacts(artifact_uri=model_uri)
print(f"Model downloaded to: {local_path}\n")

for root, dirs, files in os.walk(local_path):
    level = root.replace(local_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = ' ' * 2 * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

Model downloaded to: C:\Users\PPOBER~1\AppData\Local\Temp\tmprec0xenn\model\

/
  conda.yaml
  MLmodel
  model.pkl
  python_env.yaml
  requirements.txt


### Read MLmodel file

In [8]:
mlmodel_path = os.path.join(local_path, "MLmodel")
with open(mlmodel_path, 'r') as f:
    mlmodel_content = f.read()
    
print("=" * 60)
print("MLmodel file content:")
print("=" * 60)
print(mlmodel_content)

MLmodel file content:
artifact_path: file:///c:/Learn/github/mlpops-basics/03_mlflow_models/notebooks/mlruns/347532799678684600/models/m-5b6b4f3cc0244aa1a454417cf7220a39/artifacts
flavors:
  python_function:
    env:
      conda: conda.yaml
      virtualenv: python_env.yaml
    loader_module: mlflow.sklearn
    model_path: model.pkl
    predict_fn: predict
    python_version: 3.11.11
  sklearn:
    code: null
    pickled_model: model.pkl
    serialization_format: cloudpickle
    sklearn_version: 1.7.2
mlflow_version: 3.4.0
model_id: m-5b6b4f3cc0244aa1a454417cf7220a39
model_size_bytes: 153519
model_uuid: m-5b6b4f3cc0244aa1a454417cf7220a39
prompts: null
run_id: 929953b4db2f4eb791b7408476d192dc
utc_time_created: '2026-02-14 23:44:08.292323'



### Read conda.yaml file

In [9]:
conda_path = os.path.join(local_path, "conda.yaml")
if os.path.exists(conda_path):
    with open(conda_path, 'r') as f:
        conda_content = f.read()
    
    print("=" * 60)
    print("conda.yaml file content:")
    print("=" * 60)
    print(conda_content)

conda.yaml file content:
channels:
- conda-forge
dependencies:
- python=3.11.11
- pip<=25.2
- pip:
  - mlflow==3.4.0
  - cloudpickle==3.1.1
  - numpy==2.3.3
  - pandas==2.3.3
  - psutil==7.1.0
  - pyarrow==21.0.0
  - scikit-learn==1.7.2
  - scipy==1.16.2
name: mlflow-env



## 5B. Using Model Inspection Utilities

In [10]:
structure_info = inspect_model_structure(model_uri, verbose=True)

MLflow Model Structure: runs:/929953b4db2f4eb791b7408476d192dc/model

📁 Local path: C:\Users\PPOBER~1\AppData\Local\Temp\tmp138yprkf\model\

📄 Files in model:
  - MLmodel (748 bytes)
  - conda.yaml (262 bytes)
  - model.pkl (153,519 bytes)
  - python_env.yaml (128 bytes)
  - requirements.txt (123 bytes)

🏷️  Model flavors: ['python_function', 'sklearn']
⏰ Created: 2026-02-14 23:44:08.292323

🐍 Python version: python=3.11.11

📦 Requirements (8 packages):
  - mlflow==3.4.0
  - cloudpickle==3.1.1
  - numpy==2.3.3
  - pandas==2.3.3
  - psutil==7.1.0
  ... and 3 more



### Export Model to models/ Directory

In [11]:
exported_path = export_model(
    model_uri=model_uri,
    output_dir="../models",
    model_name="iris_basic_rf"
)

print(f"\nExported model location: {exported_path}")
print(f"\nYou can now use this model independently:")
print(f"  model = mlflow.sklearn.load_model('{exported_path}')")

✅ Model exported to: ../models\iris_basic_rf

Exported model location: ../models\iris_basic_rf

You can now use this model independently:
  model = mlflow.sklearn.load_model('../models\iris_basic_rf')


## 6. Compare Predictions

In [12]:
original_pred = model.predict(X_test)
sklearn_loaded_pred = loaded_model.predict(X_test)
pyfunc_loaded_pred = loaded_pyfunc.predict(X_test_df)

print("Checking prediction identity:")
print(f"Original vs sklearn.load_model: {np.array_equal(original_pred, sklearn_loaded_pred)}")
print(f"Original vs pyfunc.load_model: {np.array_equal(original_pred, pyfunc_loaded_pred)}")

print("\nClassification Report:")
print(classification_report(y_test, original_pred, target_names=target_names))

Checking prediction identity:
Original vs sklearn.load_model: True
Original vs pyfunc.load_model: True

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



c:\Learn\github\mlpops-basics\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


## Conclusions

In this notebook we learned:

1. ✅ Log scikit-learn model using `mlflow.sklearn.log_model()`
2. ✅ Load model in two ways:
   - As sklearn model (`mlflow.sklearn.load_model()`)
   - As Python Function (`mlflow.pyfunc.load_model()`)
3. ✅ Explore MLflow Model structure
4. ✅ Understand MLmodel file contents
5. ✅ Export models for production use

### Next Steps:
- **Notebook 02**: Model Signatures and Input Examples
- **Notebook 03**: Different Model Flavors